# 01 · Queues and backfill: FIFO vs EASY backfill on a GPU cluster

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/cluster-scheduling/01_backfill_scheduler.ipynb) &nbsp; 📖 Concept deck: [Scheduling a GPU cluster](https://nilesh-salpe.github.io/visual-ai/cluster-scheduling/01-scheduling-basics/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** a strict FIFO scheduler, an EASY backfill scheduler that lets short jobs jump ahead without delaying the job at the head of the queue, and the mean-wait metric to compare them.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU in plain Python, so no GPU or cluster is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


# A job: {"id": str, "gpus": int, "runtime": hours}. Lists are in priority order (the queue).
# Jobs never ask for more GPUs than the cluster has, and the scheduler knows each job's runtime
# (in a real cluster it only knows the time limit the user requested).

def finished(running, t):
    """Running jobs still busy at time t: list of (end_time, gpus)."""
    return [(e, g) for e, g in running if e > t]

## Task 1: Strict FIFO

Simulate a cluster of `total_gpus` GPUs, all jobs submitted at time 0. Jobs start **strictly in queue order**: a job may start only when it fits **and** every job ahead of it has started. Return `{job_id: start_time}`.

Event loop: at time `t`, drop finished jobs, start jobs from the front of the queue while they fit, then jump `t` to the next job end.

In [ ]:
def fifo_schedule(jobs, total_gpus):
    queue, running, starts, t = list(jobs), [], {}, 0
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_fifo_schedule(f):
    jobs = [dict(id="A", gpus=6, runtime=10), dict(id="B", gpus=8, runtime=5), dict(id="C", gpus=2, runtime=4), dict(id="D", gpus=2, runtime=12)]
    s = f(jobs, 8)
    assert s == {"A": 0, "B": 10, "C": 15, "D": 15}, f"got {s}: C and D must wait behind B, even though 2 GPUs sit idle for 10 hours"
    assert f([dict(id="x", gpus=4, runtime=1), dict(id="y", gpus=4, runtime=1)], 8) == {"x": 0, "y": 0}

check("fifo_schedule", _test_fifo_schedule, fifo_schedule)

## Task 2: EASY backfill

Same loop, plus backfill. When the head job does not fit:
1. **Reserve** for it: walk the running jobs in order of end time, adding their GPUs to the free count, until the head fits. That end time is the **shadow time**; the GPUs left over at that moment are the **extra** GPUs.
2. Any later job that fits **now** may start if it **finishes by the shadow time**, or if it uses only **extra** GPUs (then reduce `extra` by its GPUs).

This never delays the head job's reserved start. Return `{job_id: start_time}`.

In [ ]:
def easy_backfill(jobs, total_gpus):
    queue, running, starts, t = list(jobs), [], {}, 0
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_easy_backfill(f):
    jobs = [dict(id="A", gpus=6, runtime=10), dict(id="B", gpus=8, runtime=5), dict(id="C", gpus=2, runtime=4), dict(id="D", gpus=2, runtime=12)]
    s = f(jobs, 8)
    assert s == {"A": 0, "C": 0, "B": 10, "D": 15}, f"got {s}: C fits in the gap and ends before B's reserved start; D is too long"
    s = f(jobs, 10)
    assert s == {"A": 0, "C": 0, "D": 0, "B": 10}, f"got {s}: with 10 GPUs, 2 are extra at B's start, so the long job D may run now"

check("easy_backfill", _test_easy_backfill, easy_backfill)

## Task 3: Measure the queue

All jobs arrive at time 0, so a job's wait is its start time. Return the mean wait in hours, and the cluster **utilisation** over the makespan: GPU-hours used ÷ (total GPUs × time until the last job ends). Return `(mean_wait, utilisation)`.

In [ ]:
def mean_wait(starts, jobs, total_gpus):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_mean_wait(f):
    jobs = [dict(id="A", gpus=6, runtime=10), dict(id="B", gpus=8, runtime=5), dict(id="C", gpus=2, runtime=4), dict(id="D", gpus=2, runtime=12)]
    w, u = f({"A": 0, "B": 10, "C": 15, "D": 15}, jobs, 8)
    assert abs(w - 10.0) < 1e-9 and abs(u - 132 / (8 * 27)) < 1e-9, (w, u)
    w, u = f({"A": 0, "C": 0, "B": 10, "D": 15}, jobs, 8)
    assert abs(w - 6.25) < 1e-9, w

check("mean_wait", _test_mean_wait, mean_wait)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Strict FIFO
def fifo_schedule_solution(jobs, total_gpus):
    queue, running, starts, t = list(jobs), [], {}, 0
    while queue:
        running = finished(running, t)
        free = total_gpus - sum(g for _, g in running)
        while queue and queue[0]["gpus"] <= free:
            j = queue.pop(0)
            starts[j["id"]] = t
            running.append((t + j["runtime"], j["gpus"]))
            free -= j["gpus"]
        if queue:
            t = min(e for e, _ in running if e > t)
    return starts

check("fifo_schedule (reference)", _test_fifo_schedule, fifo_schedule_solution)

In [ ]:
# @title Solution for Task 2: EASY backfill
def easy_backfill_solution(jobs, total_gpus):
    queue, running, starts, t = list(jobs), [], {}, 0
    def start(j, free):
        starts[j["id"]] = t
        running.append((t + j["runtime"], j["gpus"]))
        return free - j["gpus"]
    while queue:
        running = finished(running, t)
        free = total_gpus - sum(g for _, g in running)
        while queue and queue[0]["gpus"] <= free:
            free = start(queue.pop(0), free)
        if not queue:
            break
        head, avail, shadow = queue[0], free, t
        for e, g in sorted(running):
            if avail >= head["gpus"]:
                break
            avail, shadow = avail + g, e
        extra = avail - head["gpus"]
        for j in list(queue[1:]):
            ends_in_time = t + j["runtime"] <= shadow
            if j["gpus"] <= free and (ends_in_time or j["gpus"] <= extra):
                free = start(j, free)
                queue.remove(j)
                if not ends_in_time:
                    extra -= j["gpus"]
        t = min(e for e, _ in running if e > t)
    return starts

check("easy_backfill (reference)", _test_easy_backfill, easy_backfill_solution)

In [ ]:
# @title Solution for Task 3: Measure the queue
def mean_wait_solution(starts, jobs, total_gpus):
    by_id = {j["id"]: j for j in jobs}
    wait = sum(starts.values()) / len(starts)
    makespan = max(starts[k] + by_id[k]["runtime"] for k in starts)
    used = sum(j["gpus"] * j["runtime"] for j in jobs)
    return wait, used / (total_gpus * makespan)

check("mean_wait (reference)", _test_mean_wait, mean_wait_solution)

---
## Experiments
A random workload of many small jobs and a few large ones on 64 GPUs: how much does backfill cut waiting? (It uses the true runtimes; with padded time limits it backfills less.)

In [ ]:
import random
random.seed(0)
work = [dict(id=f"j{i}", gpus=random.choice([1, 1, 2, 4, 8, 8, 16, 32, 64]), runtime=random.choice([1, 2, 4, 8, 24])) for i in range(300)]
for name, sched in [("FIFO", fifo_schedule_solution), ("EASY backfill", easy_backfill_solution)]:
    w, u = mean_wait_solution(sched(work, 64), work, 64)
    print(f"{name:14} mean wait {w:7.1f} h   utilisation {u:.0%}")

---
## Takeaways
* Strict FIFO leaves GPUs idle behind a large job at the head of the queue.
* **Backfill** fills those holes with jobs that cannot delay the reserved start of the head job, so it depends on accurate **time limits**.
* This is the idea behind Slurm's backfill plugin (`bf_window`, `bf_interval`), which also re-plans as jobs end early.